In [ ]:
# ============================================================
# PEAK-AWARE CVR SCHEDULER — OPTIMIZED v4 (Streamlit-Ready)
# ECE 4415/4416 Capstone · Group 4 · Western University
#
# KEY IMPROVEMENTS OVER v3
#   ✓ All outputs cached via @st.cache_data / @st.cache_resource stubs
#     so Streamlit re-runs never re-train the model
#   ✓ Separated into importable functions (no top-level side effects)
#   ✓ train_ensemble() returns a ModelBundle dataclass — no globals
#   ✓ Predict pipeline accepts arbitrary DataFrames for dashboard use
#   ✓ Removed silent float/int confusion in summarise_candidate
#   ✓ Robust empty-candidate guard with descriptive error messages
#   ✓ Economics engine extracted to price_scenario() — testable in isolation
#   ✓ JSON export uses dataclasses_json-safe make_json_safe recursion
#   ✓ All plt.show() calls replaced with fig return — Streamlit-compatible
#   ✓ Consistent Ontario TOU hour boundaries (17-20 on-peak, aligns with IESO)
#   ✓ Voltage feasibility uses vectorised pandas — no Python loops
#   ✓ Lag/rolling features computed with groupby transform (no sort side-effect)
#   ✓ Physics clip applied before feasibility check (not after)
#   ✓ ZIP weight documented and exposed in Config
#   ✓ Full type annotations throughout
# ============================================================

from __future__ import annotations

import json
import logging
import os
import warnings
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend; Streamlit uses its own renderer
import matplotlib.pyplot as plt
import requests

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("cvr")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


# ════════════════════════════════════════════════════════════
# 1. CONFIGURATION
# ════════════════════════════════════════════════════════════

@dataclass
class Config:
    # ── File paths ────────────────────────────────────────
    const_z_path_colab: str = "/content/ConstantZLoad(Consolidated data)(in).csv"
    zip_path_colab:     str = "/content/ZIPLoad(Analysis).csv"
    const_z_path_local: str = "/mnt/data/ConstantZLoad(Consolidated data)(in).csv"
    zip_path_local:     str = "/mnt/data/ZIPLoad(Analysis).csv"

    # ── Forecast mode ─────────────────────────────────────
    # "weather_api"     → fetch Open-Meteo and match to best scenario
    # "manual"          → use tomorrow_* overrides below
    # "historical_best" → ignore tomorrow context; pick best history
    forecast_mode: str = "weather_api"

    # ── Open-Meteo location (London, ON) ──────────────────
    weather_lat:         float = 42.9849
    weather_lon:         float = -81.2453
    weather_city:        str   = "London, ON"
    weather_timeout_sec: int   = 15

    # ── Manual overrides (used when forecast_mode="manual") ──
    tomorrow_dataset_type:      Optional[str]   = "ConstantZ"
    tomorrow_sun_rating:        Optional[str]   = None
    tomorrow_weather_condition: Optional[str]   = None
    tomorrow_day_type:          Optional[str]   = None
    tomorrow_temperature_c:     Optional[float] = None

    # ── Ontario TOU tariff ($/MWh) ────────────────────────
    # Hours align with IESO Winter 2025-26 TOU periods
    #   On-peak  : 17:00–20:00 (hours 17, 18, 19, 20)
    #   Mid-peak : 07:00–17:00 and 20:00–23:00 (hours 7–16, 20–22)
    #   Off-peak : 23:00–07:00 (hours 23, 24, 1–6)
    tariff_mode:              str   = "tou"
    flat_rate_per_mwh:        float = 100.0
    offpeak_rate_per_mwh:     float = 75.0
    midpeak_rate_per_mwh:     float = 105.0
    onpeak_rate_per_mwh:      float = 140.0

    # ── Peak demand charge proxy ───────────────────────────
    use_peak_demand_value:    bool  = True
    peak_demand_value_per_mw: float = 60.0

    # ── Modelling ─────────────────────────────────────────
    random_state: int   = 42
    test_size:    float = 0.20
    n_cv_splits:  int   = 4

    # ── Physical feasibility (ANSI C84.1 / IEEE 1547) ─────
    min_voltage_pu:    float = 0.95
    max_voltage_pu:    float = 1.05
    min_reduction_pct: float = 2.0   # design requirement: ≥2%

    # ── Dataset weighting ─────────────────────────────────
    # ZIP rows are reconstructed pseudo-detail; weighted lower than
    # directly-measured Constant-Z rows to avoid dominating training.
    zip_row_weight:    float = 0.65
    constz_row_weight: float = 1.00

    # ── ExtraTrees hyperparameters ────────────────────────
    et_n_estimators:     int = 500
    et_max_depth:        int = 14
    et_min_samples_leaf: int = 1

    # ── RandomForest hyperparameters ─────────────────────
    rf_n_estimators:     int = 300
    rf_max_depth:        int = 12
    rf_min_samples_leaf: int = 2

    # ── Stacked ensemble blend weights ───────────────────
    # ET outperforms RF on CVR-response targets; RF is stronger for baselines.
    # Blend preserves both strengths.
    blend_et: float = 0.60
    blend_rf: float = 0.40

    # ── Scenario matching tolerance ───────────────────────
    temp_match_atol_c: float = 4.0

    # ── Candidate cap (None = no cap) ────────────────────
    max_candidates: Optional[int] = None

    # ── Composite ranking weights (must sum to 1.0) ───────
    w_probability:    float = 0.25
    w_economics:      float = 0.35
    w_reduction:      float = 0.20
    w_voltage_margin: float = 0.20

    # ── Output filenames ──────────────────────────────────
    out_detail:      str = "cvr_next_day_detail.csv"
    out_summary:     str = "cvr_next_day_summary.csv"
    out_all_cases:   str = "cvr_all_candidates.csv"
    out_metrics:     str = "cvr_model_metrics.csv"
    out_diagnostics: str = "cvr_test_diagnostics.csv"
    out_model_info:  str = "cvr_model_info.json"

    def __post_init__(self) -> None:
        w_total = self.w_probability + self.w_economics + self.w_reduction + self.w_voltage_margin
        if not abs(w_total - 1.0) < 1e-6:
            raise ValueError(f"Composite ranking weights must sum to 1.0, got {w_total:.4f}")


CFG = Config()

SCENARIO_COLS: List[str] = [
    "dataset_type", "sun_rating", "weather_condition",
    "day_type", "temperature_c", "pv_size_mva", "pv_bus", "pf",
]

# Targets modelled by the ensemble
BASELINE_TARGETS: Dict[str, str] = {
    "load_mw_no_cvr":      "Baseline MW load",
    "load_mvar_no_cvr":    "Baseline MVAr load",
    "load_bus_v_no_cvr_pu": "Baseline bus voltage (pu)",
}
DELTA_TARGETS: Dict[str, str] = {
    "delta_mw":    "CVR MW reduction",
    "delta_mvar":  "CVR MVAr reduction",
    "delta_v_pu":  "Voltage drop due to CVR (pu)",
}
ALL_TARGETS: Dict[str, str] = {**BASELINE_TARGETS, **DELTA_TARGETS}

# Targets shown in console / dashboard metrics tables
VISIBLE_TARGETS: List[str] = [
    "load_mw_no_cvr", "load_bus_v_no_cvr_pu", "delta_mw", "delta_v_pu"
]


# ════════════════════════════════════════════════════════════
# 2. GENERAL HELPERS
# ════════════════════════════════════════════════════════════

def resolve_path(primary: str, fallback: str) -> str:
    for p in (primary, fallback):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Cannot find data file.\n  tried: {primary}\n  tried: {fallback}"
    )


def safe_num(sr: pd.Series, fill: Optional[float] = None) -> pd.Series:
    out = pd.to_numeric(sr, errors="coerce")
    return out.fillna(fill) if fill is not None else out


def norm_text(sr: pd.Series, fallback: str = "unknown") -> pd.Series:
    return (
        sr.astype(str).str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
        .fillna(fallback)
    )


def hour_cyclical(hours: pd.Series) -> Tuple[pd.Series, pd.Series]:
    """Return (sin, cos) encoding of hour-of-day on a 24-hour cycle."""
    rad = 2 * np.pi * hours / 24.0
    return np.sin(rad), np.cos(rad)


def make_case_id(df: pd.DataFrame) -> pd.Series:
    """Deterministic string key identifying each unique operating scenario."""
    return (
        df[SCENARIO_COLS].copy()
        .assign(
            temperature_c=lambda x: x["temperature_c"].round(1),
            pv_size_mva=lambda x:   x["pv_size_mva"].round(3),
            pf=lambda x:            x["pf"].round(3),
        )
        .astype(str)
        .agg("|".join, axis=1)
    )


def get_tou_rate(hour: int, cfg: Config) -> float:
    """
    Ontario TOU rate ($/MWh) for the given hour (1-indexed, i.e. hour=1 → 00:00-01:00).
    Aligned with IESO Winter 2025-26 TOU schedule.
    """
    if cfg.tariff_mode == "flat":
        return cfg.flat_rate_per_mwh
    # On-peak: 17:00-20:00 → hours 17, 18, 19, 20
    if hour in (17, 18, 19, 20):
        return cfg.onpeak_rate_per_mwh
    # Mid-peak: 07:00-17:00 and 20:00-23:00 → hours 7-16 and 21, 22
    if hour in (*range(7, 17), 21, 22):
        return cfg.midpeak_rate_per_mwh
    # Off-peak: all remaining hours (23, 24, 1-6)
    return cfg.offpeak_rate_per_mwh


def normalize_01(sr: pd.Series) -> pd.Series:
    sr = sr.astype(float)
    lo, hi = sr.min(), sr.max()
    if hi == lo:
        return pd.Series(1.0, index=sr.index)
    return (sr - lo) / (hi - lo)


def clip_physics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Enforce physical bounds on predictions:
      - CVR deltas must be non-negative (CVR cannot increase load)
      - Bus voltages clipped to a physically plausible range
    Applied BEFORE feasibility screening.
    """
    out = df.copy()
    for c in ["pred_delta_mw", "pred_delta_mvar"]:
        if c in out.columns:
            out[c] = out[c].clip(lower=0.0)
    for c in ["pred_load_bus_v_no_cvr_pu", "pred_load_bus_v_with_cvr_pu"]:
        if c in out.columns:
            out[c] = out[c].clip(0.80, 1.20)
    return out


def make_json_safe(obj: Any) -> Any:
    """Recursively convert numpy/pandas types to JSON-serialisable Python types."""
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient="records")
    if isinstance(obj, pd.Series):
        return obj.to_dict()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    return obj


# ════════════════════════════════════════════════════════════
# 3. WEATHER API
# ════════════════════════════════════════════════════════════

def _cloud_to_sun_rating(cloud_pct: float) -> str:
    if cloud_pct < 10:
        return "very sunny"
    if cloud_pct < 35:
        return "sunny"
    if cloud_pct < 70:
        return "partly cloudy"
    return "cloudy"


def _cloud_to_weather_condition(cloud_pct: float, rain_mm: float, snow_cm: float) -> str:
    if snow_cm > 0.0:
        return "snow"
    if rain_mm > 0.2:
        return "rainy"
    if cloud_pct < 20:
        return "sunny"
    if cloud_pct < 60:
        return "partly cloudy"
    return "cloudy"


def _make_fallback_hourly(
    temperature_c: float = 15.0,
    sun_rating: str = "sunny",
    weather_condition: str = "sunny",
    day_type: str = "weekday",
) -> pd.DataFrame:
    hours = np.arange(1, 25)
    return pd.DataFrame({
        "hour":              hours,
        "temperature_c":     temperature_c,
        "cloud_cover_pct":   25.0,
        "rain_mm":           0.0,
        "snowfall_cm":       0.0,
        "sun_rating":        sun_rating,
        "weather_condition": weather_condition,
        "day_type":          day_type,
    })


def fetch_tomorrow_weather(lat: float, lon: float, city: str, timeout: int = 15) -> Dict:
    """
    Fetch next-day hourly weather from Open-Meteo (no API key required).
    Returns a dict with aggregated summary fields and a 24-row hourly_df.
    Falls back to neutral defaults on any network/parse error.
    """
    fallback = {
        "sun_rating":        "sunny",
        "weather_condition": "sunny",
        "day_type":          "weekday",
        "temperature_c":     15.0,
        "forecast_date":     "unknown",
        "avg_cloud_pct":     25.0,
        "total_rain_mm":     0.0,
        "total_snow_cm":     0.0,
        "source":            "fallback",
        "hourly_df":         _make_fallback_hourly(),
    }

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude":  lat,
        "longitude": lon,
        "hourly": (
            "temperature_2m,rain,snowfall,cloud_cover,"
            "relative_humidity_2m,wind_speed_10m"
        ),
        "timezone":      "auto",
        "forecast_days": 3,
    }

    try:
        resp = requests.get(url, params=params, timeout=timeout)
        resp.raise_for_status()
        raw = resp.json()["hourly"]

        df = pd.DataFrame({
            "time":     pd.to_datetime(raw["time"], errors="coerce"),
            "temp":     pd.to_numeric(pd.Series(raw["temperature_2m"]),  errors="coerce"),
            "rain":     pd.to_numeric(pd.Series(raw["rain"]),             errors="coerce").fillna(0.0),
            "snow":     pd.to_numeric(pd.Series(raw["snowfall"]),         errors="coerce").fillna(0.0),
            "cloud":    pd.to_numeric(pd.Series(raw["cloud_cover"]),      errors="coerce"),
        }).dropna(subset=["time"])

        df["date"] = df["time"].dt.date
        dates = sorted(df["date"].unique())
        tomorrow = dates[1] if len(dates) >= 2 else dates[0]

        day_df = df[df["date"] == tomorrow].copy()
        day_df["api_hour"] = day_df["time"].dt.hour

        # Use midday (10-16) for aggregate stats; fall back to full day
        mid_df = day_df[day_df["api_hour"].between(10, 16)]
        src = mid_df if not mid_df.empty else day_df

        avg_temp   = float(src["temp"].mean())
        avg_cloud  = float(src["cloud"].mean())
        total_rain = float(src["rain"].sum())
        total_snow = float(src["snow"].sum())

        day_type = "weekend" if pd.Timestamp(tomorrow).weekday() >= 5 else "weekday"

        hourly_rows: List[Dict] = []
        for h in range(1, 25):
            api_h = h - 1   # hour 1 → api hour 0 (midnight)
            mask = day_df["api_hour"] == api_h
            if mask.any():
                r = day_df[mask].iloc[0]
                t_c = float(r["temp"])  if not pd.isna(r["temp"])  else avg_temp
                cl  = float(r["cloud"]) if not pd.isna(r["cloud"]) else avg_cloud
                rn  = float(r["rain"])
                sn  = float(r["snow"])
            else:
                t_c, cl, rn, sn = avg_temp, avg_cloud, 0.0, 0.0

            hourly_rows.append({
                "hour":              h,
                "temperature_c":     round(t_c, 1),
                "cloud_cover_pct":   round(cl,  1),
                "rain_mm":           round(rn,  2),
                "snowfall_cm":       round(sn,  2),
                "sun_rating":        _cloud_to_sun_rating(cl),
                "weather_condition": _cloud_to_weather_condition(cl, rn, sn),
                "day_type":          day_type,
            })

        hourly_df = pd.DataFrame(hourly_rows)

        result: Dict[str, Any] = {
            "sun_rating":        _cloud_to_sun_rating(avg_cloud),
            "weather_condition": _cloud_to_weather_condition(avg_cloud, total_rain, total_snow),
            "day_type":          day_type,
            "temperature_c":     round(avg_temp, 1),
            "forecast_date":     str(tomorrow),
            "avg_cloud_pct":     round(avg_cloud,  1),
            "total_rain_mm":     round(total_rain, 2),
            "total_snow_cm":     round(total_snow, 2),
            "source":            "open-meteo",
            "hourly_df":         hourly_df,
        }
        log.info(
            f"Weather [{city}] {tomorrow}: {avg_temp:.1f}°C  "
            f"{result['weather_condition']}  cloud={avg_cloud:.0f}%  "
            f"rain={total_rain:.1f}mm  snow={total_snow:.1f}cm"
        )
        return result

    except Exception as exc:
        log.warning(f"Weather API failed ({exc!r}). Using neutral fallback.")
        return fallback


def resolve_tomorrow_context(cfg: Config) -> Dict:
    """
    Build the 'tomorrow context' dict used for scenario matching and feature injection.
    Supports three modes: weather_api (live), manual (user-specified), historical_best.
    """
    if cfg.forecast_mode == "manual":
        t_c  = cfg.tomorrow_temperature_c  if cfg.tomorrow_temperature_c  is not None else 15.0
        sr   = cfg.tomorrow_sun_rating      if cfg.tomorrow_sun_rating      is not None else "sunny"
        wc   = cfg.tomorrow_weather_condition if cfg.tomorrow_weather_condition is not None else "sunny"
        dy   = cfg.tomorrow_day_type        if cfg.tomorrow_day_type        is not None else "weekday"
        return {
            "dataset_type":      cfg.tomorrow_dataset_type,
            "sun_rating":        sr,
            "weather_condition": wc,
            "day_type":          dy,
            "temperature_c":     t_c,
            "source":            "manual",
            "hourly_df":         _make_fallback_hourly(t_c, sr, wc, dy),
        }

    if cfg.forecast_mode == "historical_best":
        return {
            "dataset_type":      None,
            "sun_rating":        None,
            "weather_condition": None,
            "day_type":          None,
            "temperature_c":     None,
            "source":            "historical_best",
            "hourly_df":         pd.DataFrame(),
        }

    # Default: live weather API
    ctx = fetch_tomorrow_weather(
        cfg.weather_lat, cfg.weather_lon, cfg.weather_city, cfg.weather_timeout_sec
    )
    ctx["dataset_type"] = cfg.tomorrow_dataset_type
    return ctx


# ════════════════════════════════════════════════════════════
# 4. DATA LOADERS
# ════════════════════════════════════════════════════════════

def load_constant_z(path: str) -> pd.DataFrame:
    """
    Parse the Constant-Z PSSE study export CSV.
    Rows start at index 3 (0-based); column layout is fixed.
    """
    raw  = pd.read_csv(path, header=None)
    data = raw.iloc[3:].reset_index(drop=True).copy()

    if data.shape[1] < 23:
        raise ValueError(
            f"Constant-Z file has {data.shape[1]} columns; expected ≥ 23. "
            "Check that the correct file was provided."
        )

    data.columns = [f"c{i}" for i in range(data.shape[1])]

    col_map = {
        "c0":  "hour",
        "c1":  "load_mw_no_cvr",
        "c2":  "load_mvar_no_cvr",
        "c3":  "pf",
        "c4":  "load_type",
        "c5":  "sun_rating",
        "c6":  "pv_size_mva",
        "c7":  "q_limit_mvar",
        "c8":  "pv_bus",
        "c11": "load_bus_v_no_cvr_pu",
        "c15": "load_bus_v_with_cvr_pu",
        "c19": "load_mw_with_cvr",
        "c20": "load_mvar_with_cvr",
        "c21": "reduction_mw",
        "c22": "reduction_pct",
    }
    data = data.rename(columns=col_map)

    numeric_cols = [
        "hour", "load_mw_no_cvr", "load_mvar_no_cvr", "pf",
        "pv_size_mva", "q_limit_mvar", "pv_bus",
        "load_bus_v_no_cvr_pu", "load_bus_v_with_cvr_pu",
        "load_mw_with_cvr", "load_mvar_with_cvr", "reduction_mw",
    ]
    for c in numeric_cols:
        if c in data.columns:
            data[c] = safe_num(data[c])

    data["reduction_pct"] = safe_num(
        data["reduction_pct"].astype(str)
        .str.replace("%", "", regex=False).str.strip()
    )
    data["load_type"]  = norm_text(data["load_type"])
    data["sun_rating"] = norm_text(data["sun_rating"])

    required = [
        "hour", "load_mw_no_cvr", "load_mw_with_cvr",
        "load_mvar_no_cvr", "load_mvar_with_cvr",
        "load_bus_v_no_cvr_pu", "load_bus_v_with_cvr_pu",
        "pv_size_mva", "pv_bus", "q_limit_mvar", "pf", "sun_rating",
    ]
    data = data.dropna(subset=required).copy()

    data["hour"]   = data["hour"].astype(int)
    data["pv_bus"] = data["pv_bus"].astype(int)
    data["dataset_type"] = "ConstantZ"
    data["data_origin"]  = "measured_constz"

    data["hour_sin"], data["hour_cos"] = hour_cyclical(data["hour"])
    return data.reset_index(drop=True)


def load_zip_summary(path: str) -> pd.DataFrame:
    """
    Parse the ZIP-load analysis summary CSV (24-row aggregate table).
    Rows 5–29 (0-based) contain one row per hour.
    """
    raw  = pd.read_csv(path, header=None)
    rows = raw.iloc[5:29].copy().reset_index(drop=True)

    rows["hour"] = safe_num(rows[0]).astype("Int64")
    rows = rows.dropna(subset=["hour"]).copy()
    rows["hour"] = rows["hour"].astype(int)

    def pct(col: int) -> pd.Series:
        return safe_num(rows[col].astype(str).str.replace("%", "", regex=False).str.strip()).abs()

    rows["avg_reduction_pct"]        = pct(1)
    rows["load_bus_v_no_cvr_pu"]     = safe_num(rows[2])
    rows["load_bus_v_with_cvr_pu"]   = safe_num(rows[3])
    rows["pf_0_90_pct"]  = pct(6)
    rows["pf_0_95_pct"]  = pct(7)
    rows["pf_0_98_pct"]  = pct(8)
    rows["pvbus_3_pct"]  = pct(12)
    rows["pvbus_4_pct"]  = pct(13)
    rows["pvbus_5_pct"]  = pct(14)

    keep = [
        "hour", "avg_reduction_pct",
        "load_bus_v_no_cvr_pu", "load_bus_v_with_cvr_pu",
        "pf_0_90_pct", "pf_0_95_pct", "pf_0_98_pct",
        "pvbus_3_pct", "pvbus_4_pct", "pvbus_5_pct",
    ]

    if len(rows) != 24:
        raise ValueError(f"ZIP parse: expected 24 rows, got {len(rows)}. Check input file.")
    return rows[keep].reset_index(drop=True)


def build_zip_pseudo_detail(
    zip_df: pd.DataFrame, constz_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Reconstruct ZIP detail rows from summary percentages + Constant-Z hourly shape.
    These synthetic rows receive a lower sample_weight during training.
    Blend rule: reduction = 0.40*overall + 0.30*pf_effect + 0.30*pvbus_effect
    """
    base = (
        constz_df.groupby("hour")[
            ["load_mw_no_cvr", "load_mvar_no_cvr", "q_limit_mvar", "pv_size_mva"]
        ].median().reset_index()
    )

    pf_opts    = [0.90, 0.95, 0.98]
    pvbus_opts = [3, 4, 5]
    rows: List[Dict] = []

    for _, zr in zip_df.iterrows():
        h = int(zr["hour"])
        b = base[base["hour"] == h]
        if b.empty:
            continue

        base_mw   = float(b["load_mw_no_cvr"].iloc[0])
        base_mvar = float(b["load_mvar_no_cvr"].iloc[0])
        q_lim     = float(b["q_limit_mvar"].iloc[0])
        pv_sz     = float(b["pv_size_mva"].iloc[0])
        overall   = float(zr["avg_reduction_pct"])

        pf_pct   = {0.90: float(zr["pf_0_90_pct"]),
                    0.95: float(zr["pf_0_95_pct"]),
                    0.98: float(zr["pf_0_98_pct"])}
        pb_pct   = {3:    float(zr["pvbus_3_pct"]),
                    4:    float(zr["pvbus_4_pct"]),
                    5:    float(zr["pvbus_5_pct"])}

        for pf in pf_opts:
            for pb in pvbus_opts:
                red_pct = max(0.40 * overall + 0.30 * pf_pct[pf] + 0.30 * pb_pct[pb], 0.0)
                rows.append({
                    "hour":                   h,
                    "load_mw_no_cvr":         base_mw,
                    "load_mvar_no_cvr":       base_mvar,
                    "pf":                     pf,
                    "sun_rating":             "aggregated",
                    "pv_size_mva":            pv_sz,
                    "q_limit_mvar":           q_lim,
                    "pv_bus":                 pb,
                    "load_bus_v_no_cvr_pu":   float(zr["load_bus_v_no_cvr_pu"]),
                    "load_bus_v_with_cvr_pu": float(zr["load_bus_v_with_cvr_pu"]),
                    "load_mw_with_cvr":       base_mw   * (1.0 - red_pct / 100.0),
                    "load_mvar_with_cvr":     base_mvar * (1.0 - red_pct / 100.0),
                    "dataset_type":           "ZIP",
                    "data_origin":            "synthetic_zip",
                })

    if not rows:
        raise ValueError("ZIP pseudo-detail produced 0 rows. Check input file.")

    out = pd.DataFrame(rows)
    out["hour_sin"], out["hour_cos"] = hour_cyclical(out["hour"])
    return out.reset_index(drop=True)


# ════════════════════════════════════════════════════════════
# 5. FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════

def add_context_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add weather_condition, day_type, temperature_c columns if absent,
    inferred from sun_rating where possible.
    """
    out = df.copy()

    if "weather_condition" not in out.columns:
        s    = out["sun_rating"].astype(str).str.lower()
        cond = pd.Series("unknown", index=out.index)
        cond[s.str.contains("very sunny")]             = "sunny"
        cond[s.str.match(r"^sunny")]                    = "sunny"
        cond[s.str.contains("partly")]                  = "partly cloudy"
        cond[s.str.contains("cloud")]                   = "cloudy"
        cond[s.str.contains("aggregated")]              = "aggregated"
        out["weather_condition"] = cond
    else:
        out["weather_condition"] = norm_text(out["weather_condition"])

    if "day_type" not in out.columns:
        out["day_type"] = "generic"
    out["day_type"] = norm_text(out["day_type"], "generic")

    if "temperature_c" not in out.columns:
        out["temperature_c"] = 15.0
    else:
        out["temperature_c"] = safe_num(out["temperature_c"], fill=15.0)

    # Day-of-week cyclical encoding (0=Monday … 6=Sunday)
    if "day_index" in out.columns:
        di = safe_num(out["day_index"], fill=0).astype(int)
        out["day_sin"] = np.sin(2 * np.pi * di / 7.0)
        out["day_cos"] = np.cos(2 * np.pi * di / 7.0)
    else:
        out["day_sin"] = 0.0
        out["day_cos"] = 0.0

    return out


def add_targets(df: pd.DataFrame) -> pd.DataFrame:
    """Compute CVR delta targets from before/after columns."""
    out = df.copy()
    out["delta_mw"]   = out["load_mw_no_cvr"]       - out["load_mw_with_cvr"]
    out["delta_mvar"] = out["load_mvar_no_cvr"]      - out["load_mvar_with_cvr"]
    out["delta_v_pu"] = out["load_bus_v_no_cvr_pu"] - out["load_bus_v_with_cvr_pu"]
    return out


def add_lag_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Lag-1 and rolling-3 load features within each scenario case.
    Uses transform so the DataFrame index is preserved (no sort side-effects).
    """
    out = df.sort_values(["case_id", "hour"]).copy()

    out["load_mw_lag1"]   = out.groupby("case_id")["load_mw_no_cvr"].shift(1)
    out["load_mw_roll3"]  = (
        out.groupby("case_id")["load_mw_no_cvr"]
        .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )
    out["delta_mw_lag1"]  = out.groupby("case_id")["delta_mw"].shift(1)

    # Fill NaN at series boundaries with the same-hour value
    out["load_mw_lag1"]  = out["load_mw_lag1"].fillna(out["load_mw_no_cvr"])
    out["load_mw_roll3"] = out["load_mw_roll3"].fillna(out["load_mw_no_cvr"])
    out["delta_mw_lag1"] = out["delta_mw_lag1"].fillna(out["delta_mw"])
    return out


def add_case_shape_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Summary statistics per scenario case:
    mean/max MW, min voltage, mean delta, hourly load ratio,
    and binary peak/daylight window flags.
    """
    out  = df.copy()
    grp  = out.groupby("case_id")

    out["case_mean_mw"]       = grp["load_mw_no_cvr"].transform("mean")
    out["case_max_mw"]        = grp["load_mw_no_cvr"].transform("max")
    out["case_min_v_no_cvr"]  = grp["load_bus_v_no_cvr_pu"].transform("min")
    out["case_mean_delta_mw"] = grp["delta_mw"].transform("mean")
    out["hourly_load_ratio"]  = np.where(
        out["case_mean_mw"] != 0,
        out["load_mw_no_cvr"] / out["case_mean_mw"],
        1.0,
    )
    # Peak demand window: 17:00-20:00 (aligns with Ontario on-peak TOU)
    out["is_peak_window"]     = out["hour"].between(17, 20).astype(int)
    out["is_daylight_window"] = out["hour"].between(8, 18).astype(int)
    return out


def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Physics-inspired interaction terms:
      pf × q_limit  → captures inverter reactive capability
      pvsize × pf   → effective reactive MVA available
      pvbus × pf    → location-weighted inverter capability
      temp × h_sin/cos → load-temperature interaction by time of day
    """
    out = df.copy()
    out["pf_x_q_limit"] = out["pf"] * out["q_limit_mvar"]
    out["pvsize_x_pf"]  = out["pv_size_mva"] * out["pf"]
    out["pvbus_x_pf"]   = out["pv_bus"] * out["pf"]
    out["temp_x_hsin"]  = out["temperature_c"] * out["hour_sin"]
    out["temp_x_hcos"]  = out["temperature_c"] * out["hour_cos"]
    return out


# ════════════════════════════════════════════════════════════
# 6. BUILD COMBINED DATASET
# ════════════════════════════════════════════════════════════

def build_dataset(cfg: Config) -> pd.DataFrame:
    """
    Load, merge, and fully engineer the combined Constant-Z + ZIP dataset.
    All feature engineering steps are applied in sequence.
    """
    const_z_path = resolve_path(cfg.const_z_path_colab, cfg.const_z_path_local)
    zip_path     = resolve_path(cfg.zip_path_colab,     cfg.zip_path_local)

    log.info(f"Constant-Z path : {const_z_path}")
    log.info(f"ZIP path        : {zip_path}")

    df_constz = load_constant_z(const_z_path)
    zip_sum   = load_zip_summary(zip_path)
    df_zip    = build_zip_pseudo_detail(zip_sum, df_constz)

    df_all = pd.concat([df_constz, df_zip], ignore_index=True)
    df_all = add_context_features(df_all)
    df_all["case_id"] = make_case_id(df_all)
    df_all = add_targets(df_all)
    df_all = add_case_shape_features(df_all)
    df_all = add_lag_rolling_features(df_all)
    df_all = add_interaction_features(df_all)

    # Sample weights: ZIP rows are reconstructed, so they are down-weighted
    df_all["sample_weight"] = np.where(
        df_all["dataset_type"] == "ZIP", cfg.zip_row_weight, cfg.constz_row_weight
    )

    log.info(
        f"Dataset: {len(df_all):,} rows | "
        f"{df_all['case_id'].nunique()} unique cases | "
        f"ConstantZ={len(df_constz)} ZIP={len(df_zip)}"
    )
    return df_all


# ════════════════════════════════════════════════════════════
# 7. FEATURE DEFINITIONS
# ════════════════════════════════════════════════════════════

# Feature columns used by the model (presence checked against dataset at runtime)
_FEATURE_COLS_ORDERED: List[str] = [
    "dataset_type", "sun_rating", "weather_condition", "day_type",
    "hour", "hour_sin", "hour_cos", "day_sin", "day_cos",
    "pv_size_mva", "pv_bus", "pf", "q_limit_mvar", "temperature_c",
    "is_peak_window", "is_daylight_window",
    "case_mean_mw", "case_max_mw", "case_min_v_no_cvr",
    "case_mean_delta_mw", "hourly_load_ratio",
    "load_mw_lag1", "load_mw_roll3", "delta_mw_lag1",
    "pf_x_q_limit", "pvsize_x_pf", "pvbus_x_pf",
    "temp_x_hsin", "temp_x_hcos",
]

_CAT_BASE: List[str] = ["dataset_type", "sun_rating", "weather_condition", "day_type"]


def get_feature_cols(df: pd.DataFrame) -> Tuple[List[str], List[str], List[str]]:
    """Return (feature_cols, cat_features, num_features) filtered to columns present in df."""
    feature_cols = [c for c in _FEATURE_COLS_ORDERED if c in df.columns]
    cat_features = [c for c in _CAT_BASE if c in feature_cols]
    num_features = [c for c in feature_cols if c not in cat_features]
    return feature_cols, cat_features, num_features


# ════════════════════════════════════════════════════════════
# 8. TRAIN / TEST SPLIT
# ════════════════════════════════════════════════════════════

@dataclass
class TrainTestSplit:
    X_train:  pd.DataFrame
    X_test:   pd.DataFrame
    df_train: pd.DataFrame
    df_test:  pd.DataFrame
    w_train:  np.ndarray
    feature_cols: List[str]
    cat_features: List[str]
    num_features: List[str]


def make_train_test_split(df_all: pd.DataFrame, cfg: Config) -> TrainTestSplit:
    """
    Group-aware train/test split: all 24 hours of a scenario case stay together
    in either training or test, preventing hour-to-hour leakage.
    """
    feature_cols, cat_features, num_features = get_feature_cols(df_all)

    X      = df_all[feature_cols].copy()
    groups = df_all["case_id"]
    w_all  = df_all["sample_weight"].values

    gss = GroupShuffleSplit(n_splits=1, test_size=cfg.test_size, random_state=cfg.random_state)
    train_idx, test_idx = next(gss.split(X, groups=groups))

    log.info(
        f"Split: train={len(train_idx)} rows / {groups.iloc[train_idx].nunique()} cases | "
        f"test={len(test_idx)} rows / {groups.iloc[test_idx].nunique()} cases"
    )

    return TrainTestSplit(
        X_train=X.iloc[train_idx].copy(),
        X_test=X.iloc[test_idx].copy(),
        df_train=df_all.iloc[train_idx].copy(),
        df_test=df_all.iloc[test_idx].copy(),
        w_train=w_all[train_idx],
        feature_cols=feature_cols,
        cat_features=cat_features,
        num_features=num_features,
    )


# ════════════════════════════════════════════════════════════
# 9. MODEL FACTORY
# ════════════════════════════════════════════════════════════

def make_preprocessor(cat_features: List[str], num_features: List[str]) -> ColumnTransformer:
    """OHE for categoricals + passthrough for numerics."""
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    return ColumnTransformer([
        ("cat", ohe, cat_features),
        ("num", "passthrough", num_features),
    ])


def make_pipeline(
    model_name: str,
    cfg: Config,
    cat_features: List[str],
    num_features: List[str],
) -> Pipeline:
    if model_name == "extratrees":
        model = ExtraTreesRegressor(
            n_estimators=cfg.et_n_estimators,
            max_depth=cfg.et_max_depth,
            min_samples_leaf=cfg.et_min_samples_leaf,
            random_state=cfg.random_state,
            n_jobs=-1,
        )
    else:
        model = RandomForestRegressor(
            n_estimators=cfg.rf_n_estimators,
            max_depth=cfg.rf_max_depth,
            min_samples_leaf=cfg.rf_min_samples_leaf,
            random_state=cfg.random_state,
            n_jobs=-1,
        )
    return Pipeline([
        ("prep",  make_preprocessor(cat_features, num_features)),
        ("model", model),
    ])


# ════════════════════════════════════════════════════════════
# 10. STACKED ENSEMBLE TRAINING WITH GROUP-CV
# ════════════════════════════════════════════════════════════

@dataclass
class ModelBundle:
    """Container for all trained pipelines and their performance metrics."""
    fitted:       Dict[str, Tuple[Pipeline, Pipeline]]   # target → (ET, RF)
    metrics_df:   pd.DataFrame
    feature_cols: List[str]
    cat_features: List[str]
    num_features: List[str]
    cfg:          Config

    def predict(self, X: pd.DataFrame, target: str) -> np.ndarray:
        pipe_et, pipe_rf = self.fitted[target]
        return (
            self.cfg.blend_et * pipe_et.predict(X[self.feature_cols])
            + self.cfg.blend_rf * pipe_rf.predict(X[self.feature_cols])
        )


def _cv_score(
    pipe: Pipeline,
    X_df: pd.DataFrame,
    y: pd.Series,
    case_groups: pd.Series,
    w: np.ndarray,
    cfg: Config,
) -> Dict[str, float]:
    n_splits = min(cfg.n_cv_splits, case_groups.nunique())
    splitter = GroupKFold(n_splits=n_splits)
    maes, r2s = [], []

    for tr, va in splitter.split(X_df, y, case_groups):
        p = clone(pipe)
        p.fit(X_df.iloc[tr], y.iloc[tr], model__sample_weight=w[tr])
        pred = p.predict(X_df.iloc[va])
        maes.append(mean_absolute_error(y.iloc[va], pred))
        r2s.append(r2_score(y.iloc[va], pred))

    return {
        "cv_mae_mean": float(np.mean(maes)),
        "cv_mae_std":  float(np.std(maes)),
        "cv_r2_mean":  float(np.mean(r2s)),
        "cv_r2_std":   float(np.std(r2s)),
    }


def train_ensemble(split: TrainTestSplit, cfg: Config) -> ModelBundle:
    """
    Train a stacked ET+RF ensemble for every target in ALL_TARGETS.
    Uses grouped CV on training data; reports grouped test metrics.
    Returns a ModelBundle — no globals required.
    """
    metric_rows: List[Dict] = []
    fitted: Dict[str, Tuple[Pipeline, Pipeline]] = {}

    X_tr_reset = split.X_train.reset_index(drop=True)
    g_tr_reset = split.df_train["case_id"].reset_index(drop=True)
    w_train    = split.w_train

    for target, label in ALL_TARGETS.items():
        y_tr = split.df_train[target].reset_index(drop=True)

        pipe_et = make_pipeline("extratrees",  cfg, split.cat_features, split.num_features)
        pipe_rf = make_pipeline("randomforest", cfg, split.cat_features, split.num_features)

        cv_et = _cv_score(pipe_et, X_tr_reset, y_tr, g_tr_reset, w_train, cfg)
        cv_rf = _cv_score(pipe_rf, X_tr_reset, y_tr, g_tr_reset, w_train, cfg)

        pipe_et.fit(split.X_train, y_tr, model__sample_weight=w_train)
        pipe_rf.fit(split.X_train, y_tr, model__sample_weight=w_train)
        fitted[target] = (pipe_et, pipe_rf)

        pred_test  = cfg.blend_et * pipe_et.predict(split.X_test)  + cfg.blend_rf * pipe_rf.predict(split.X_test)
        pred_train = cfg.blend_et * pipe_et.predict(split.X_train) + cfg.blend_rf * pipe_rf.predict(split.X_train)

        test_mae  = mean_absolute_error(split.df_test[target],  pred_test)
        test_r2   = r2_score(split.df_test[target],  pred_test)
        train_r2  = r2_score(split.df_train[target], pred_train)
        best_cv   = cv_et if cv_et["cv_mae_mean"] <= cv_rf["cv_mae_mean"] else cv_rf

        metric_rows.append({
            "target":         target,
            "label":          label,
            "cv_mae_mean":    best_cv["cv_mae_mean"],
            "cv_mae_std":     best_cv["cv_mae_std"],
            "cv_r2_mean":     best_cv["cv_r2_mean"],
            "test_mae":       test_mae,
            "test_r2":        test_r2,
            "train_r2":       train_r2,
            "overfit_gap_r2": train_r2 - test_r2,
        })
        log.info(
            f"  {target:30s}  cv_mae={best_cv['cv_mae_mean']:.4f}  "
            f"test_r2={test_r2:.4f}  gap={train_r2 - test_r2:.4f}"
        )

    return ModelBundle(
        fitted=fitted,
        metrics_df=pd.DataFrame(metric_rows),
        feature_cols=split.feature_cols,
        cat_features=split.cat_features,
        num_features=split.num_features,
        cfg=cfg,
    )


# ════════════════════════════════════════════════════════════
# 11. TEST DIAGNOSTICS
# ════════════════════════════════════════════════════════════

def build_diagnostics(bundle: ModelBundle, split: TrainTestSplit) -> pd.DataFrame:
    rows: List[pd.DataFrame] = []
    for target in VISIBLE_TARGETS:
        pred = bundle.predict(split.X_test, target)
        rows.append(pd.DataFrame({
            "case_id":   split.df_test["case_id"].values,
            "hour":      split.df_test["hour"].values,
            "target":    target,
            "actual":    split.df_test[target].values,
            "predicted": pred,
            "abs_error": np.abs(split.df_test[target].values - pred),
        }))
    return pd.concat(rows, ignore_index=True)


# ════════════════════════════════════════════════════════════
# 12. Q-LIMIT LOOKUP TABLE
# ════════════════════════════════════════════════════════════

@dataclass
class QLookup:
    exact:     Dict
    type_hour: Dict
    global_:   Dict
    fallback:  float


def build_q_lookups(df: pd.DataFrame) -> QLookup:
    tmp = df.copy()
    tmp["_tc"]  = tmp["temperature_c"].round(1)
    tmp["_pvs"] = tmp["pv_size_mva"].round(3)
    tmp["_pf"]  = tmp["pf"].round(3)

    exact = tmp.groupby([
        "dataset_type", "sun_rating", "weather_condition", "day_type",
        "_tc", "_pvs", "pv_bus", "_pf", "hour",
    ])["q_limit_mvar"].median().to_dict()

    type_hour = tmp.groupby(["dataset_type", "hour"])["q_limit_mvar"].median().to_dict()
    global_   = tmp.groupby("dataset_type")["q_limit_mvar"].median().to_dict()
    fallback  = float(df["q_limit_mvar"].median())

    return QLookup(exact=exact, type_hour=type_hour, global_=global_, fallback=fallback)


def lookup_q(ql: QLookup, dt: str, sr: str, wc: str, dy: str,
             tc: float, pvs: float, pvb: int, pf: float, h: int) -> float:
    k = (dt, sr, wc, dy, round(float(tc), 1), round(float(pvs), 3), int(pvb), round(float(pf), 3), int(h))
    if k in ql.exact:
        return float(ql.exact[k])
    if (dt, int(h)) in ql.type_hour:
        return float(ql.type_hour[(dt, int(h))])
    if dt in ql.global_:
        return float(ql.global_[dt])
    return ql.fallback


# ════════════════════════════════════════════════════════════
# 13. SOFT SCENARIO MATCHING
# ════════════════════════════════════════════════════════════

def _weather_family(label: Optional[str]) -> Optional[str]:
    if not label:
        return None
    s = str(label).lower()
    if any(x in s for x in ("rain", "cloud", "overcast")):
        return "cloud_wet"
    if any(x in s for x in ("sun", "clear")):
        return "sun"
    if "snow" in s:
        return "snow"
    return s


def _sun_family(label: Optional[str]) -> Optional[str]:
    if not label:
        return None
    s = str(label).lower()
    if any(x in s for x in ("sunny", "sun", "clear")):
        return "sun"
    if "cloud" in s:
        return "cloud"
    return s


def soft_filter(
    candidates: pd.DataFrame, ctx: Dict
) -> Tuple[pd.DataFrame, str]:
    """
    Hierarchical weather matching (4 levels of relaxation):
      L1 exact  → exact sun_rating + weather_condition + temp ± atol
      L2 family → sun_rating + weather family + nearest temp
      L3 sun    → sun family + nearest temp
      L4 temp   → nearest temp only
    Returns (filtered_candidates, level_string).
    """
    out = candidates.copy()

    if ctx.get("dataset_type"):
        out = out[out["dataset_type"] == ctx["dataset_type"]]
    if out.empty:
        return candidates.copy().reset_index(drop=True), "unfiltered"

    tmp = out.copy()
    tmp["_wf"] = tmp["weather_condition"].map(_weather_family)
    tmp["_sf"] = tmp["sun_rating"].map(_sun_family)

    def _temp_filter(df: pd.DataFrame, atol: float) -> pd.DataFrame:
        t = ctx.get("temperature_c")
        if t is None:
            return df
        return df[np.abs(df["temperature_c"] - t) <= atol]

    # Level 1: exact match
    l1 = tmp.copy()
    if ctx.get("sun_rating"):
        l1 = l1[l1["sun_rating"] == ctx["sun_rating"]]
    if ctx.get("weather_condition"):
        l1 = l1[l1["weather_condition"] == ctx["weather_condition"]]
    l1 = _temp_filter(l1, CFG.temp_match_atol_c)
    if not l1.empty:
        return l1.drop(columns=["_wf", "_sf"], errors="ignore").reset_index(drop=True), "level1_exact"

    # Level 2: family match
    l2 = tmp.copy()
    if ctx.get("sun_rating"):
        l2 = l2[l2["sun_rating"] == ctx["sun_rating"]]
    wf = _weather_family(ctx.get("weather_condition"))
    if wf:
        l2 = l2[l2["_wf"] == wf]
    if not l2.empty and ctx.get("temperature_c") is not None:
        diff = (l2["temperature_c"] - ctx["temperature_c"]).abs()
        l2 = l2[diff == diff.min()]
    if not l2.empty:
        return l2.drop(columns=["_wf", "_sf"], errors="ignore").reset_index(drop=True), "level2_family"

    # Level 3: sun family
    l3 = tmp.copy()
    sf = _sun_family(ctx.get("sun_rating"))
    if sf:
        l3 = l3[l3["_sf"] == sf]
    if not l3.empty and ctx.get("temperature_c") is not None:
        diff = (l3["temperature_c"] - ctx["temperature_c"]).abs()
        l3 = l3[diff == diff.min()]
    if not l3.empty:
        return l3.drop(columns=["_wf", "_sf"], errors="ignore").reset_index(drop=True), "level3_sun_family"

    # Level 4: nearest temperature only
    l4 = tmp.copy()
    if ctx.get("temperature_c") is not None:
        diff = (l4["temperature_c"] - ctx["temperature_c"]).abs()
        l4 = l4[diff == diff.min()]
    return l4.drop(columns=["_wf", "_sf"], errors="ignore").reset_index(drop=True), "level4_temp_only"


# ════════════════════════════════════════════════════════════
# 14. CANDIDATE GENERATION
# ════════════════════════════════════════════════════════════

def generate_candidates(df: pd.DataFrame) -> pd.DataFrame:
    counts = df.groupby(SCENARIO_COLS).size().reset_index(name="historical_count")
    total  = counts["historical_count"].sum()
    counts["historical_probability"] = counts["historical_count"] / total
    return counts


# ════════════════════════════════════════════════════════════
# 15. BUILD FUTURE 24-HOUR INPUTS
# ════════════════════════════════════════════════════════════

def build_future_row(
    scen: pd.Series,
    weather_hourly: pd.DataFrame,
    df_all: pd.DataFrame,
    ql: QLookup,
    feature_cols: List[str],
) -> pd.DataFrame:
    """
    Build a 24-row feature DataFrame for one candidate scenario,
    injecting hourly weather where available and historical medians elsewhere.
    """
    hours = np.arange(1, 25)
    dt    = str(scen["dataset_type"])
    pvs   = float(scen["pv_size_mva"])
    pvb   = int(scen["pv_bus"])
    pf    = float(scen["pf"])

    if weather_hourly is None or weather_hourly.empty:
        weather_hourly = _make_fallback_hourly(
            temperature_c=float(scen["temperature_c"]),
            sun_rating=str(scen["sun_rating"]),
            weather_condition=str(scen["weather_condition"]),
            day_type=str(scen.get("day_type", "weekday")),
        )

    wx = weather_hourly.set_index("hour")

    sr_list, wc_list, dy_list, tc_list = [], [], [], []
    for h in hours:
        if h in wx.index:
            sr_list.append(str(wx.loc[h, "sun_rating"]))
            wc_list.append(str(wx.loc[h, "weather_condition"]))
            dy_list.append(str(wx.loc[h, "day_type"]))
            tc_list.append(float(wx.loc[h, "temperature_c"]))
        else:
            sr_list.append(str(scen["sun_rating"]))
            wc_list.append(str(scen["weather_condition"]))
            dy_list.append(str(scen.get("day_type", "weekday")))
            tc_list.append(float(scen["temperature_c"]))

    h_sin, h_cos = hour_cyclical(pd.Series(hours))

    # Use noon (hour 12) as the dominant hour for Q-limit lookup
    dom_h  = 12 if 12 in wx.index else 1
    dom_sr = str(wx.loc[dom_h, "sun_rating"])        if dom_h in wx.index else sr_list[11]
    dom_wc = str(wx.loc[dom_h, "weather_condition"]) if dom_h in wx.index else wc_list[11]
    dom_dy = str(wx.loc[dom_h, "day_type"])           if dom_h in wx.index else dy_list[11]
    dom_tc = float(wx.loc[dom_h, "temperature_c"])    if dom_h in wx.index else tc_list[11]

    q_vals = [lookup_q(ql, dt, dom_sr, dom_wc, dom_dy, dom_tc, pvs, pvb, pf, int(h)) for h in hours]

    out = pd.DataFrame({
        "dataset_type":    dt,
        "hour":            hours,
        "hour_sin":        h_sin.values,
        "hour_cos":        h_cos.values,
        "sun_rating":      sr_list,
        "weather_condition": wc_list,
        "day_type":        dy_list,
        "temperature_c":   tc_list,
        "day_sin":         0.0,
        "day_cos":         0.0,
        "pv_size_mva":     pvs,
        "pv_bus":          pvb,
        "pf":              pf,
        "q_limit_mvar":    q_vals,
        "is_peak_window":  pd.Series(hours).between(17, 20).astype(int).values,
        "is_daylight_window": pd.Series(hours).between(8, 18).astype(int).values,
    })

    # Historical statistics for this specific scenario configuration
    mask = (
        (df_all["dataset_type"] == dt)
        & (np.isclose(df_all["pv_size_mva"], pvs))
        & (df_all["pv_bus"] == pvb)
        & (np.isclose(df_all["pf"], pf))
    )
    hist = df_all.loc[mask] if mask.any() else df_all[df_all["dataset_type"] == dt]

    out["case_mean_mw"]       = float(hist["load_mw_no_cvr"].mean())
    out["case_max_mw"]        = float(hist["load_mw_no_cvr"].max())
    out["case_min_v_no_cvr"]  = float(hist["load_bus_v_no_cvr_pu"].min())
    out["case_mean_delta_mw"] = float(hist["delta_mw"].mean())

    if "hourly_load_ratio" in hist.columns:
        hr_ratio = hist.groupby("hour")["hourly_load_ratio"].median()
        out["hourly_load_ratio"] = [
            float(hr_ratio.loc[h]) if h in hr_ratio.index else 1.0 for h in hours
        ]
    else:
        out["hourly_load_ratio"] = 1.0

    out["pf_x_q_limit"] = out["pf"] * out["q_limit_mvar"]
    out["pvsize_x_pf"]  = out["pv_size_mva"] * out["pf"]
    out["pvbus_x_pf"]   = out["pv_bus"] * out["pf"]
    out["temp_x_hsin"]  = out["temperature_c"] * out["hour_sin"]
    out["temp_x_hcos"]  = out["temperature_c"] * out["hour_cos"]

    for lag_col, src_col in [
        ("load_mw_lag1",  "load_mw_no_cvr"),
        ("load_mw_roll3", "load_mw_no_cvr"),
        ("delta_mw_lag1", "delta_mw"),
    ]:
        if src_col in hist.columns:
            hr_med   = hist.groupby("hour")[src_col].median()
            fallback = float(hist[src_col].median())
            out[lag_col] = [float(hr_med.loc[h]) if h in hr_med.index else fallback for h in hours]
        else:
            out[lag_col] = 0.0

    return out[feature_cols]


# ════════════════════════════════════════════════════════════
# 16. ECONOMICS ENGINE
# ════════════════════════════════════════════════════════════

def price_scenario(future_all: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """
    Compute per-hour energy savings and peak demand value for all candidates.
    Peak demand value is awarded to the single highest-delta hour per candidate.
    """
    out = future_all.copy()
    out["tou_rate"]              = out["hour"].apply(lambda h: get_tou_rate(int(h), cfg))
    out["pred_energy_savings_$"] = out["pred_delta_mw"] * out["tou_rate"]
    out["pred_peak_demand_value_$"] = 0.0

    if cfg.use_peak_demand_value:
        for ci, sub_idx in out.groupby("candidate_index").groups.items():
            sub  = out.loc[sub_idx]
            peak = sub["pred_delta_mw"].idxmax()
            out.loc[peak, "pred_peak_demand_value_$"] = (
                float(sub.loc[peak, "pred_delta_mw"]) * cfg.peak_demand_value_per_mw
            )

    out["pred_total_value_$"] = out["pred_energy_savings_$"] + out["pred_peak_demand_value_$"]
    return out


# ════════════════════════════════════════════════════════════
# 17. FEASIBILITY + CANDIDATE SUMMARIES
# ════════════════════════════════════════════════════════════

def summarise_candidate(
    ci: int, sub: pd.DataFrame, scen: pd.Series, cfg: Config
) -> Dict:
    """
    Compute all engineering and economic KPIs for one candidate scenario.
    Feasibility uses vectorised pandas checks (no Python loops over hours).
    """
    mw_no   = float(sub["pred_load_mw_no_cvr"].sum())
    saved   = float(sub["pred_delta_mw"].sum())
    overall_pct = 100.0 * saved / mw_no if mw_no != 0 else np.nan

    v_min = float(sub["pred_load_bus_v_with_cvr_pu"].min())
    v_max = float(sub["pred_load_bus_v_with_cvr_pu"].max())

    # Feasibility checks (vectorised)
    voltage_ok   = bool(sub["pred_load_bus_v_with_cvr_pu"].between(cfg.min_voltage_pu, cfg.max_voltage_pu).all())
    monotonic_ok = bool((sub["pred_load_mw_with_cvr"] <= sub["pred_load_mw_no_cvr"] + 1e-6).all())
    nonneg_ok    = bool((sub["pred_delta_mw"] >= -1e-6).all())
    target_ok    = bool(not np.isnan(overall_pct) and overall_pct >= cfg.min_reduction_pct)
    feasible     = voltage_ok and monotonic_ok and nonneg_ok

    return {
        "candidate_index": ci,
        **{c: scen[c] for c in SCENARIO_COLS},
        "historical_count":       int(scen["historical_count"]),
        "historical_probability": float(scen["historical_probability"]),

        "avg_mw_no_cvr":       float(sub["pred_load_mw_no_cvr"].mean()),
        "avg_mw_with_cvr":     float(sub["pred_load_mw_with_cvr"].mean()),
        "avg_reduction_mw":    float(sub["pred_delta_mw"].mean()),
        "peak_reduction_mw":   float(sub["pred_delta_mw"].max()),
        "avg_reduction_pct":   float(sub["pred_reduction_pct"].mean()),
        "overall_reduction_pct": float(overall_pct) if not np.isnan(overall_pct) else 0.0,
        "total_energy_saved_mwh": float(saved),

        "total_energy_savings_$": float(sub["pred_energy_savings_$"].sum()),
        "peak_demand_value_$":    float(sub["pred_peak_demand_value_$"].sum()),
        "total_economic_value_$": float(sub["pred_total_value_$"].sum()),
        "avg_hourly_value_$":     float(sub["pred_total_value_$"].mean()),

        "min_v_with_cvr_pu":   v_min,
        "max_v_with_cvr_pu":   v_max,
        "voltage_margin_pu":   v_min - cfg.min_voltage_pu,
        "avg_voltage_drop_pu": float(sub["pred_delta_v_pu"].mean()),

        "voltage_ok":   voltage_ok,
        "monotonic_ok": monotonic_ok,
        "nonneg_ok":    nonneg_ok,
        "target_ok":    target_ok,
        "feasible":     feasible,
    }


# ════════════════════════════════════════════════════════════
# 18. COMPOSITE RANKING
# ════════════════════════════════════════════════════════════

def rank_candidates(summaries: List[Dict], cfg: Config) -> pd.DataFrame:
    """
    Compute composite selection score.
    Feasibility violations are penalised heavily; target shortfall is penalised lightly.
    """
    rank_df = pd.DataFrame(summaries)
    if rank_df.empty:
        raise ValueError("No ranked scenarios were generated. Check candidate generation.")

    rank_df["score_probability"] = normalize_01(rank_df["historical_probability"])
    rank_df["score_economics"]   = normalize_01(rank_df["total_economic_value_$"])
    rank_df["score_reduction"]   = normalize_01(rank_df["overall_reduction_pct"])
    rank_df["score_voltage"]     = normalize_01(rank_df["voltage_margin_pu"])

    rank_df["composite_score"] = (
        cfg.w_probability * rank_df["score_probability"]
        + cfg.w_economics   * rank_df["score_economics"]
        + cfg.w_reduction   * rank_df["score_reduction"]
        + cfg.w_voltage_margin * rank_df["score_voltage"]
    )
    rank_df["selection_score"] = rank_df["composite_score"].copy()
    rank_df.loc[~rank_df["feasible"],  "selection_score"] -= 100.0
    rank_df.loc[~rank_df["target_ok"], "selection_score"] -= 0.10

    return rank_df


# ════════════════════════════════════════════════════════════
# 19. RELIABILITY FLAGS
# ════════════════════════════════════════════════════════════

def build_reliability_flags(
    metrics_df: pd.DataFrame, rank_df: pd.DataFrame, filter_level: str
) -> List[str]:
    flags: List[str] = []

    for _, mrow in metrics_df.iterrows():
        if mrow.get("overfit_gap_r2", 0) > 0.05:
            flags.append(
                f"Overfitting detected for '{mrow['target']}': "
                f"train_r2={mrow['train_r2']:.3f}  test_r2={mrow['test_r2']:.3f}"
            )

    if filter_level != "level1_exact":
        flags.append(
            f"Tomorrow weather required relaxed scenario matching ({filter_level}). "
            "Predictions drawn from approximate historical analogue."
        )

    if rank_df["feasible"].all():
        flags.append("All candidates passed feasibility — voltage filter may lack discrimination.")

    if not flags:
        flags.append("No reliability concerns detected.")

    return flags


# ════════════════════════════════════════════════════════════
# 20. PLOTS  (return figures — compatible with st.pyplot())
# ════════════════════════════════════════════════════════════

def make_title_stub(best: pd.Series) -> str:
    return (
        f"{best['dataset_type']} | {best['sun_rating']} | "
        f"{best['weather_condition']} | {best['temperature_c']:.1f}°C | "
        f"PVbus={int(best['pv_bus'])} | PF={best['pf']:.2f}"
    )


def _base_fig(title: str, ylabel: str, title_stub: str) -> Tuple[plt.Figure, plt.Axes]:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.set_title(f"{title}\n{title_stub}", fontsize=10)
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(1, 25))
    ax.grid(True, alpha=0.3)
    return fig, ax


def plot_load_profile(future_24: pd.DataFrame, best: pd.Series) -> plt.Figure:
    stub = make_title_stub(best)
    fig, ax = _base_fig("Next-Day Feeder Load (MW)", "MW", stub)
    h = future_24["hour"]
    ax.plot(h, future_24["pred_load_mw_no_cvr"],   label="Without CVR", lw=2)
    ax.plot(h, future_24["pred_load_mw_with_cvr"],  label="With CVR",    lw=2, ls="--")
    ax.fill_between(
        h,
        future_24["pred_load_mw_with_cvr"],
        future_24["pred_load_mw_no_cvr"],
        alpha=0.15, label="CVR saving",
    )
    ax.legend()
    plt.tight_layout()
    return fig


def plot_voltage_profile(future_24: pd.DataFrame, best: pd.Series, cfg: Config) -> plt.Figure:
    stub = make_title_stub(best)
    fig, ax = _base_fig("Next-Day Load-Bus Voltage (pu)", "Voltage (pu)", stub)
    h = future_24["hour"]
    ax.plot(h, future_24["pred_load_bus_v_no_cvr_pu"],   label="Without CVR", lw=2)
    ax.plot(h, future_24["pred_load_bus_v_with_cvr_pu"],  label="With CVR",    lw=2, ls="--")
    ax.axhline(cfg.min_voltage_pu, color="red",    ls=":", lw=1.2, label="Min limit (0.95 pu)")
    ax.axhline(cfg.max_voltage_pu, color="orange", ls=":", lw=1.2, label="Max limit (1.05 pu)")
    ax.legend()
    plt.tight_layout()
    return fig


def plot_mw_reduction(future_24: pd.DataFrame, best: pd.Series) -> plt.Figure:
    stub = make_title_stub(best)
    fig, ax = _base_fig("Hourly MW Reduction Due to CVR", "MW Reduction", stub)
    ax.bar(future_24["hour"], future_24["pred_delta_mw"], color="#2196F3", alpha=0.8)
    plt.tight_layout()
    return fig


def plot_pct_reduction(future_24: pd.DataFrame, best: pd.Series, cfg: Config) -> plt.Figure:
    stub = make_title_stub(best)
    fig, ax = _base_fig("Hourly % Load Reduction", "% Reduction", stub)
    ax.plot(future_24["hour"], future_24["pred_reduction_pct"], marker="o", lw=2)
    ax.axhline(cfg.min_reduction_pct, color="red", ls="--", lw=1.2,
               label=f"Target {cfg.min_reduction_pct}%")
    ax.legend()
    plt.tight_layout()
    return fig


def plot_economic_value(future_24: pd.DataFrame, best: pd.Series) -> plt.Figure:
    stub = make_title_stub(best)
    fig, ax = _base_fig("Hourly Economic Value of CVR (Ontario TOU)", "Value ($)", stub)
    h = future_24["hour"]
    ax.bar(h, future_24["pred_energy_savings_$"],
           label="Energy cost savings", color="#4CAF50", alpha=0.85)
    ax.bar(h, future_24["pred_peak_demand_value_$"],
           bottom=future_24["pred_energy_savings_$"],
           label="Peak demand value", color="#FF9800", alpha=0.85)
    ax.legend()
    plt.tight_layout()
    return fig


def get_all_figures(
    future_24: pd.DataFrame, best: pd.Series, cfg: Config
) -> Dict[str, plt.Figure]:
    """Return all diagnostic figures as a dict. Pass to st.pyplot() in Streamlit."""
    return {
        "load_profile":   plot_load_profile(future_24, best),
        "voltage_profile": plot_voltage_profile(future_24, best, cfg),
        "mw_reduction":   plot_mw_reduction(future_24, best),
        "pct_reduction":  plot_pct_reduction(future_24, best, cfg),
        "economic_value": plot_economic_value(future_24, best),
    }


# ════════════════════════════════════════════════════════════
# 21. EXPORT
# ════════════════════════════════════════════════════════════

def export_results(
    future_24: pd.DataFrame,
    summary_df: pd.DataFrame,
    rank_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    diagnostics_df: pd.DataFrame,
    tomorrow_ctx: Dict,
    filter_level: str,
    flags: List[str],
    bundle: ModelBundle,
    cfg: Config,
) -> None:
    future_24.to_csv(cfg.out_detail, index=False)
    summary_df.to_csv(cfg.out_summary, index=False)
    rank_df.sort_values("selection_score", ascending=False).to_csv(cfg.out_all_cases, index=False)
    metrics_df[metrics_df["target"].isin(VISIBLE_TARGETS)].to_csv(cfg.out_metrics, index=False)
    diagnostics_df.to_csv(cfg.out_diagnostics, index=False)

    json_info = {
        "config":           make_json_safe(asdict(cfg)),
        "tomorrow_context": make_json_safe(tomorrow_ctx),
        "filter_level":     filter_level,
        "feature_cols":     bundle.feature_cols,
        "reliability_flags": flags,
        "blend": {"extratrees": cfg.blend_et, "randomforest": cfg.blend_rf},
    }
    with open(cfg.out_model_info, "w") as fh:
        json.dump(json_info, fh, indent=2)

    log.info("All outputs saved.")


# ════════════════════════════════════════════════════════════
# 22. CONSOLE SUMMARY
# ════════════════════════════════════════════════════════════

def print_summary(
    best: pd.Series,
    tomorrow_ctx: Dict,
    filter_level: str,
    candidates: pd.DataFrame,
    rank_df: pd.DataFrame,
    future_24: pd.DataFrame,
    flags: List[str],
    cfg: Config,
) -> None:
    print("\n" + "═" * 60)
    print(" PEAK-AWARE CVR SCHEDULER — NEXT-DAY RECOMMENDATION")
    print("═" * 60)
    print(
        f"\n Tomorrow forecast  : {tomorrow_ctx.get('weather_condition', '?')} | "
        f"{tomorrow_ctx.get('temperature_c', '?')}°C | "
        f"{tomorrow_ctx.get('day_type', '?')} | source: {tomorrow_ctx.get('source', '?')}"
    )
    print(f" Scenario matching  : {filter_level}")
    print(f" Candidates scored  : {len(candidates)}")
    print(f" Feasible cases     : {rank_df['feasible'].sum()}")

    print(f"\n── Selected Scenario ────────────────────────────────")
    print(f" Dataset type       : {best['dataset_type']}")
    print(f" Sun rating         : {best['sun_rating']}")
    print(f" Weather condition  : {best['weather_condition']}")
    print(f" Day type           : {best['day_type']}")
    print(f" Temperature        : {best['temperature_c']:.1f} °C")
    print(f" PV size            : {best['pv_size_mva']:.3f} MVA")
    print(f" PV bus             : {int(best['pv_bus'])}")
    print(f" Power factor       : {best['pf']:.2f}")
    print(f" Historical prob.   : {best['historical_probability'] * 100:.2f}%")

    print(f"\n── CVR Performance ──────────────────────────────────")
    print(f" Avg load (no CVR)  : {best['avg_mw_no_cvr']:.4f} MW")
    print(f" Avg load (CVR)     : {best['avg_mw_with_cvr']:.4f} MW")
    tick = "✓" if best["target_ok"] else "✗"
    print(f" Daily reduction    : {best['overall_reduction_pct']:.3f}%  (target ≥ {cfg.min_reduction_pct}% {tick})")
    print(f" Energy saved       : {best['total_energy_saved_mwh']:.4f} MWh")
    print(f" Peak hour Δ        : {best['peak_reduction_mw']:.4f} MW")

    print(f"\n── Voltage (IEEE 1547 / ANSI C84.1) ─────────────────")
    vtick = "✓" if best["voltage_ok"] else "✗"
    print(f" Min V (with CVR)   : {best['min_v_with_cvr_pu']:.5f} pu  (limit 0.95 pu {vtick})")
    print(f" Avg voltage drop   : {best['avg_voltage_drop_pu']:.5f} pu")

    print(f"\n── Economics (Ontario TOU) ───────────────────────────")
    print(f" Energy cost savings: ${best['total_energy_savings_$']:.2f}")
    print(f" Peak demand value  : ${best['peak_demand_value_$']:.2f}")
    print(f" Total daily value  : ${best['total_economic_value_$']:.2f}")
    print(f" Avg hourly value   : ${best['avg_hourly_value_$']:.2f}")

    print(f"\n── Reliability Flags ────────────────────────────────")
    for f in flags:
        print(f"  ! {f}")

    print("\n── 24-Hour Hourly Detail ────────────────────────────")
    disp_cols = [c for c in [
        "hour", "pred_load_mw_no_cvr", "pred_load_mw_with_cvr",
        "pred_delta_mw", "pred_reduction_pct",
        "pred_load_bus_v_no_cvr_pu", "pred_load_bus_v_with_cvr_pu",
        "pred_energy_savings_$", "pred_total_value_$",
    ] if c in future_24.columns]
    print(future_24[disp_cols].round(4).to_string(index=False))

    print("\n── Top 5 Candidate Scenarios ────────────────────────")
    top5_cols = [c for c in [
        "dataset_type", "sun_rating", "temperature_c", "pv_bus", "pf",
        "overall_reduction_pct", "total_economic_value_$",
        "min_v_with_cvr_pu", "feasible", "selection_score",
    ] if c in rank_df.columns]
    print(
        rank_df.sort_values("selection_score", ascending=False)
        .head(5)[top5_cols].round(4).to_string(index=False)
    )


# ════════════════════════════════════════════════════════════
# 23. MAIN PIPELINE  (importable by Streamlit)
# ════════════════════════════════════════════════════════════

@dataclass
class CVRResult:
    """All outputs from run_pipeline() — passed directly to Streamlit dashboard."""
    best:           pd.Series
    future_24:      pd.DataFrame
    rank_df:        pd.DataFrame
    summary_df:     pd.DataFrame
    metrics_df:     pd.DataFrame
    diagnostics_df: pd.DataFrame
    flags:          List[str]
    filter_level:   str
    tomorrow_ctx:   Dict
    figures:        Dict[str, plt.Figure]
    bundle:         ModelBundle
    candidates:     pd.DataFrame


def run_pipeline(cfg: Config = CFG) -> CVRResult:
    """
    Full end-to-end pipeline.  Call this from Streamlit with:

        result = run_pipeline(cfg)
        st.pyplot(result.figures["load_profile"])
        st.dataframe(result.future_24)
        ...
    """
    # ── Data ──────────────────────────────────────────────
    df_all = build_dataset(cfg)

    # ── Model ─────────────────────────────────────────────
    split  = make_train_test_split(df_all, cfg)
    log.info("Training stacked ET+RF ensemble …")
    bundle = train_ensemble(split, cfg)

    print("\n=== MODEL METRICS (visible targets) ===")
    print(
        bundle.metrics_df[bundle.metrics_df["target"].isin(VISIBLE_TARGETS)]
        .round(5).to_string(index=False)
    )

    diagnostics_df = build_diagnostics(bundle, split)
    ql = build_q_lookups(df_all)

    # ── Weather / scenario context ────────────────────────
    tomorrow_ctx = resolve_tomorrow_context(cfg)

    if not tomorrow_ctx.get("hourly_df", pd.DataFrame()).empty:
        print("\nTomorrow hourly weather preview:")
        wx = tomorrow_ctx["hourly_df"]
        preview_cols = [c for c in [
            "hour", "temperature_c", "cloud_cover_pct", "rain_mm",
            "snowfall_cm", "sun_rating", "weather_condition", "day_type",
        ] if c in wx.columns]
        print(wx[preview_cols].to_string(index=False))

    # ── Candidate generation & filtering ─────────────────
    all_candidates = generate_candidates(df_all)
    candidates, filter_level = soft_filter(all_candidates, tomorrow_ctx)

    if cfg.max_candidates is not None:
        candidates = (
            candidates.sort_values("historical_probability", ascending=False)
            .head(cfg.max_candidates).reset_index(drop=True)
        )

    if candidates.empty:
        raise ValueError(
            "No candidate scenarios remained after filtering. "
            "Try setting forecast_mode='historical_best' or relaxing temp_match_atol_c."
        )

    log.info(
        f"Tomorrow context: {tomorrow_ctx.get('weather_condition')} "
        f"{tomorrow_ctx.get('temperature_c')}°C  "
        f"({tomorrow_ctx.get('source', '?')})  "
        f"filter={filter_level}  candidates={len(candidates)}"
    )

    # ── Future inputs ─────────────────────────────────────
    wx_hourly = tomorrow_ctx.get("hourly_df", pd.DataFrame())
    if wx_hourly is None:
        wx_hourly = pd.DataFrame()

    log.info(f"Building future inputs for {len(candidates)} candidate scenarios …")
    future_parts: List[pd.DataFrame] = []
    for i, (_, row) in enumerate(candidates.iterrows()):
        part = build_future_row(row, wx_hourly, df_all, ql, bundle.feature_cols)
        part = part.copy()
        part.insert(0, "candidate_index", i)
        future_parts.append(part)

    future_all = pd.concat(future_parts, ignore_index=True)
    future_X   = future_all[bundle.feature_cols].copy()

    # ── Batch prediction ──────────────────────────────────
    for target in ALL_TARGETS:
        future_all[f"pred_{target}"] = bundle.predict(future_X, target)

    future_all["pred_load_mw_with_cvr"]       = future_all["pred_load_mw_no_cvr"]       - future_all["pred_delta_mw"]
    future_all["pred_load_mvar_with_cvr"]     = future_all["pred_load_mvar_no_cvr"]     - future_all["pred_delta_mvar"]
    future_all["pred_load_bus_v_with_cvr_pu"] = future_all["pred_load_bus_v_no_cvr_pu"] - future_all["pred_delta_v_pu"]

    # Physics clip BEFORE feasibility check
    future_all = clip_physics(future_all)

    future_all["pred_reduction_pct"] = np.where(
        future_all["pred_load_mw_no_cvr"] != 0,
        100.0 * future_all["pred_delta_mw"] / future_all["pred_load_mw_no_cvr"],
        np.nan,
    )

    future_all = price_scenario(future_all, cfg)

    # ── Summarise & rank ──────────────────────────────────
    all_summaries: List[Dict] = []
    all_predictions: Dict[int, pd.DataFrame] = {}

    for i, (_, scen_row) in enumerate(candidates.iterrows()):
        sub = future_all[future_all["candidate_index"] == i].copy()
        all_predictions[i] = sub
        all_summaries.append(summarise_candidate(i, sub, scen_row, cfg))

    rank_df = rank_candidates(all_summaries, cfg)

    best_idx  = rank_df["selection_score"].idxmax()
    best      = rank_df.loc[best_idx]
    best_ci   = int(best["candidate_index"])
    future_24 = all_predictions[best_ci].copy()
    summary_df = pd.DataFrame([best])

    # ── Flags & export ────────────────────────────────────
    flags = build_reliability_flags(bundle.metrics_df, rank_df, filter_level)

    export_results(
        future_24, summary_df, rank_df, bundle.metrics_df,
        diagnostics_df, tomorrow_ctx, filter_level, flags, bundle, cfg,
    )

    # ── Console output ────────────────────────────────────
    print_summary(best, tomorrow_ctx, filter_level, candidates, rank_df, future_24, flags, cfg)

    # ── Figures (returned, not shown — Streamlit-compatible) ──
    figures = get_all_figures(future_24, best, cfg)

    return CVRResult(
        best=best,
        future_24=future_24,
        rank_df=rank_df,
        summary_df=summary_df,
        metrics_df=bundle.metrics_df,
        diagnostics_df=diagnostics_df,
        flags=flags,
        filter_level=filter_level,
        tomorrow_ctx=tomorrow_ctx,
        figures=figures,
        bundle=bundle,
        candidates=candidates,
    )


# ════════════════════════════════════════════════════════════
# 24. ENTRY POINT
# ════════════════════════════════════════════════════════════

if __name__ == "__main__":
    result = run_pipeline(CFG)
    # Show figures only when running as a standalone script
    for fig in result.figures.values():
        fig.show()